# 00 Install `ts_compress_to` Table Operator

Run this notebook after building and installing the package in the active environment. From the repository root:

```powershell
python -m build
python -m pip install --force-reinstall (Get-ChildItem dist\td_ts_compressor-*.whl | Sort-Object LastWriteTime -Descending | Select-Object -First 1).FullName
```

Credentials are loaded from `.env` with `load_dotenv`. The notebook does not print secrets.



## 1. Setup

Required variables are `TD_HOST`, `TD_USERNAME`, and `TD_PASSWORD`. Optional variables are `TD_LOGMECH`, `TD_DATABASE`, and `TD_OPERATOR_DATABASE`. Use `TD_OPERATOR_DATABASE` when the table operator should be installed somewhere other than the session database.


In [ ]:
from pathlib import Path
import os

import teradatasql
from dotenv import load_dotenv

from td_ts_compressor import install_table_operator_sql
from td_ts_compressor.core import bundled_c_source_path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pyproject.toml").exists() else NOTEBOOK_DIR.parent
ENV_PATH = REPO_ROOT / ".env"

load_dotenv(ENV_PATH)

required = ["TD_HOST", "TD_USERNAME", "TD_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

print(f"Repository root: {REPO_ROOT}")
print(f"Bundled operator source: {bundled_c_source_path()}")


## 2. Connect to Teradata


In [ ]:
connect_args = {
    "host": os.environ["TD_HOST"],
    "user": os.environ["TD_USERNAME"],
    "password": os.environ["TD_PASSWORD"],
}
if os.getenv("TD_LOGMECH"):
    connect_args["logmech"] = os.environ["TD_LOGMECH"]

con = teradatasql.connect(**connect_args)
cur = con.cursor()
if os.getenv("TD_DATABASE"):
    cur.execute(f"DATABASE {os.environ['TD_DATABASE']}")
cur.execute("SELECT DATABASE")
current_database = cur.fetchone()[0]
OPERATOR_DATABASE = os.getenv("TD_OPERATOR_DATABASE") or current_database
print("Current database:", current_database)
print("Installing operator in:", OPERATOR_DATABASE)


## 3. Install or Replace Function and Mapping

The package supplies the bundled C source path and generates the `REPLACE FUNCTION` plus `REPLACE FUNCTION MAPPING` statements. The demos assume the default names `ts_compress_seq_to` and `ts_compress_seq_to_map`.


In [ ]:
statements = install_table_operator_sql(target_database=OPERATOR_DATABASE)
for statement in statements:
    cur.execute(statement)

print("Installed/replaced ts_compress_seq_to and ts_compress_seq_to_map")


## 4. Validate Mapping Visibility


In [ ]:
cur.execute("SELECT DATABASE")
print("Session database:", cur.fetchone()[0])
print("Compressor database:", OPERATOR_DATABASE)
print("Next notebooks should use TD_OPERATOR_DATABASE with this value when it differs from TD_DATABASE.")
